# 04 — Ensemble con métricas OOF + Submission

Combina las predicciones de los 3 modelos con promedio ponderado.

**Requiere haber corrido antes los 3 notebooks OOF_FULL**, que generan:
- `lgbm_baseline_oof.csv` / `image_baseline_oof.csv` / `text_baseline_oof.csv` → para calcular la métrica del ensemble sobre train
- `my_team_01.csv` / `my_team_02.csv` / `my_team_03.csv` → para generar el submission final

**Outputs:**
- `ensemble_oof.csv` → predicciones del ensemble sobre train (subir a pestaña de práctica)
- `my_team_04.csv` → predicciones del ensemble sobre test (subir a competencia)

```
participant/
├── submissions/
├── scripts/
└── participant.zip
```

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Rutas
ZIP_PATH              = "/content/drive/MyDrive/participant/participant.zip"
SUBMISSIONS_DIR       = "/content/drive/MyDrive/participant/submissions"

os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

In [ ]:
# Extraer train_processed.csv para tener el precio real
# Solo lo necesitamos para calcular la métrica OOF del ensemble
csv_train = None
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    all_files = zip_ref.namelist()
    for f in all_files:
        if f.endswith("train_processed.csv"):
            zip_ref.extract(f, path="/content/")
            csv_train = os.path.join("/content", f)
            break

if not csv_train:
    raise FileNotFoundError("No se encontró train_processed.csv dentro del zip.")

train = pd.read_csv(csv_train)
print(f"Train cargado: {len(train)} filas")

Train cargado: 11840 filas


In [ ]:
# Cargar OOFs (predicciones sobre train de cada modelo)
try:
    oof_tab = pd.read_csv(os.path.join(SUBMISSIONS_DIR, "03_diego_MEJORADO_OOF_mape25.5_ridge_stack.csv"))
    oof_img = pd.read_csv(os.path.join(SUBMISSIONS_DIR, "image_clip_oof.csv"))
    oof_txt = pd.read_csv(os.path.join(SUBMISSIONS_DIR, "Lu_text_oof_01.csv"))
    print("OOFs cargados correctamente.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Asegurate de haber corrido los 3 notebooks OOF_FULL primero.")
    raise

OOFs cargados correctamente.


In [ ]:
# Cargar submissions de test (predicciones sobre test de cada modelo)
try:
    pred_tab = pd.read_csv(os.path.join(SUBMISSIONS_DIR, "03_diego_MEJORADO_mape25.5_ridge_stack.csv"))
    pred_img = pd.read_csv(os.path.join(SUBMISSIONS_DIR, "jaime_clip_04"))
    pred_txt = pd.read_csv(os.path.join(SUBMISSIONS_DIR, "Lu_text_test_01.csv"))
    print("Submissions de test cargadas correctamente.")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Asegurate de haber corrido los 3 notebooks OOF_FULL primero.")
    raise

Submissions de test cargadas correctamente.


In [ ]:
# Alinear OOFs con el train original por zpid
base_train = train[["zpid", "lastSoldPrice_hpi_adjusted"]].copy()

base_train = base_train.merge(oof_tab.rename(columns={"predicted_price": "tab"}), on="zpid", how="left")
base_train = base_train.merge(oof_img.rename(columns={"predicted_price": "img"}), on="zpid", how="left")
base_train = base_train.merge(oof_txt.rename(columns={"predicted_price": "txt"}), on="zpid", how="left")

# Rellenar posibles vacíos con mediana de cada columna
for col in ["tab", "img", "txt"]:
    if base_train[col].isnull().any():
        base_train[col] = base_train[col].fillna(base_train[col].median())

print(f"Train alineado: {len(base_train)} filas")

Train alineado: 11840 filas


In [ ]:
# Alinear predicciones de test por zpid
base_test = pred_tab.rename(columns={"predicted_price": "tab"}).copy()
base_test = base_test.merge(pred_img.rename(columns={"predicted_price": "img"}), on="zpid", how="left")
base_test = base_test.merge(pred_txt.rename(columns={"predicted_price": "txt"}), on="zpid", how="left")

for col in ["tab", "img", "txt"]:
    if base_test[col].isnull().any():
        base_test[col] = base_test[col].fillna(base_test[col].median())

print(f"Test alineado: {len(base_test)} filas")

Test alineado: 5038 filas


In [ ]:
# Pesos del ensemble
# Modificá estos valores para experimentar.
# Deben sumar 1.0
W_TAB = 0.98     # Original 0.60
W_IMG = 0.01       # Original 0.20, modificado a 0 porque este modelo predice mal
W_TXT = 0.01      # Original 0.20

assert abs(W_TAB + W_IMG + W_TXT - 1.0) < 1e-6, "Los pesos deben sumar 1.0"

In [ ]:
# Ensemble sobre OOFs
ensemble_oof = (
    W_TAB * base_train["tab"]
    + W_IMG * base_train["img"]
    + W_TXT * base_train["txt"]
)

# Para R2 necesitamos volver a log (las OOFs ya están en $)
real_price   = base_train["lastSoldPrice_hpi_adjusted"]
log_ensemble = np.log1p(ensemble_oof)
log_real     = np.log1p(real_price)

oof_r2   = r2_score(log_real, log_ensemble)
oof_mae  = mean_absolute_error(real_price, ensemble_oof)
oof_mape = np.mean(np.abs((real_price - ensemble_oof) / real_price)) * 100


print("MÉTRICAS ENSEMBLE OOF (validación honesta)")
print(f"Pesos: Tabular={W_TAB} | Imágenes={W_IMG} | Texto={W_TXT}")
print(f"R²  (log):  {oof_r2:.4f}")
print(f"MAE ($):    ${oof_mae:,.0f}")
print(f"MAPE (%):   {oof_mape:.1f}%")

MÉTRICAS ENSEMBLE OOF (validación honesta)
Pesos: Tabular=0.98 | Imágenes=0.01 | Texto=0.01
--------------------------------------------------
R²  (log):  0.7573
MAE ($):    $113,190
MAPE (%):   25.5%


In [ ]:
# Ensemble sobre test → submission final
ensemble_test = (
    W_TAB * base_test["tab"]
    + W_IMG * base_test["img"]
    + W_TXT * base_test["txt"]
)

In [ ]:
# Guardar OOF del ensemble (práctica) y submission (competencia)

# 1. OOF del ensemble
oof_df = pd.DataFrame({
    "zpid":            base_train["zpid"],
    "predicted_price": ensemble_oof,
})
oof_file = os.path.join(SUBMISSIONS_DIR, "Lu_ensemble_oof_04.csv")
oof_df.to_csv(oof_file, index=False)

# 2. Test
test_df = pd.DataFrame({
    "zpid":            base_test["zpid"],
    "predicted_price": ensemble_test,
})
test_file = os.path.join(SUBMISSIONS_DIR, "Grupo_ensemble_test_04.csv")
test_df.to_csv(test_file, index=False)

print(f"OOF guardado en:  {oof_file}  ({len(oof_df)} filas)")
print(f"Test guardado en: {test_file} ({len(test_df)} filas)")

OOF guardado en:  /content/drive/MyDrive/participant/submissions/Lu_ensemble_oof_04.csv  (11840 filas)
Test guardado en: /content/drive/MyDrive/participant/submissions/Grupo_ensemble_test_04.csv (5038 filas)
